# FT Sensor Calibration — Data Inspection

**Goal:** understand the data completely *before* fitting anything.

We are training the homemade 16-channel sensor to reproduce the uFactory sensor's wrench.
Everything downstream depends on the data being correctly aligned, zeroed and understood,
so this notebook works through nine checks in order. Run one cell at a time and read the
output — each step is designed to be inspected, not skimmed.

| Step | Question |
|---|---|
| 1 | What sessions do we have? |
| 2 | What does every column mean? |
| 3 | Are the two sensors on the same clock? |
| 4 | Are the sampling rates stable? |
| 5 | Any missing / bad / stuck data? |
| 6 | Where are the genuinely unloaded periods? |
| 7 | What are the zero offsets, and do they drift? |
| 8 | After removing offsets, does zero load read zero? |
| 9 | How much of each sensor's range did we actually use? |
| 10 | **Do the two sensors share a coordinate frame?** |

## Setup

Edit `DATA_DIR` if your CSVs live elsewhere.

In [ ]:
import csv, glob, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Auto-locate the session CSVs: check the working dir, then walk up to the repo root.
def _find_data_dir():
    for cand in [Path("."), *Path(".").resolve().parents]:
        if list(cand.glob("ft_calibration_session_*_run_metadata.csv")):
            return cand
    return Path(".")

DATA_DIR = _find_data_dir()   # <- override manually if your CSVs live elsewhere
print("DATA_DIR =", DATA_DIR.resolve())

AXES = ("fx","fy","fz","mx","my","mz")
CHANNELS = 16
CHAN_COLS = tuple(f"ch{i}" for i in range(1, CHANNELS+1))
UF_RATED = {"fx":150.0,"fy":150.0,"fz":200.0,"mx":4.0,"my":4.0,"mz":4.0}   # uFactory datasheet
OPENFT_VALID = (9000.0, 21000.0)                                            # openFT firmware valid band

def read_csv(p):
    with open(p) as fh: return list(csv.DictReader(fh))

def find_sessions(d=DATA_DIR):
    return sorted(re.search(r"session_(\d+)_run_metadata", p).group(1)
                  for p in glob.glob(str(d/"ft_calibration_session_*_run_metadata.csv")))

def stream(session, name, d=DATA_DIR):
    """name in {ufactory_raw, ufactory_calibrated, homemade_raw, homemade_calibrated}"""
    return read_csv(d/f"ft_calibration_session_{session}_{name}.csv")

def arr(rows, cols):  return np.array([[float(r[c]) for c in cols] for r in rows])
def ts_of(rows):      return np.array([float(r["ts"]) for r in rows])

SESSIONS = find_sessions()
print(f"{len(SESSIONS)} sessions found:")
for s in SESSIONS: print("  ", s)

## Step 1 — Inventory

What exists, how long each run was, and what config produced it.
Each session is one arm pose; the id is the unix time the run started.

In [ ]:
print(f"{'session':>12} {'started':>20} {'dur_s':>7} {'type':>6} {'uf_rows':>9} {'hm_rows':>9}")
for s in SESSIONS:
    meta = {r["key"]: r["value"] for r in stream(s,"run_metadata")}
    uf, hm = stream(s,"ufactory_calibrated"), stream(s,"homemade_raw")
    t = ts_of(uf)
    print(f"{s:>12} {meta.get('created_at','?'):>20} {t[-1]-t[0]:7.0f} "
          f"{meta.get('session_type','?'):>6} {len(uf):9d} {len(hm):9d}")

## Step 2 — Understand every column

Two things here are easy to get wrong and worth stating explicitly:

- **`ufactory_raw` vs `ufactory_calibrated`** differ *only* by gravity/payload compensation
  (a near-constant offset). **Both are already in N / N·m.** The ADC→Newton conversion happens
  inside the uFactory sensor and is never exposed to us. Use `ufactory_calibrated` as ground truth.
- **`homemade_calibrated`** is the *old shipped matrix* applied to the raw channels. It is the
  output of a previous fit, not an independent measurement — **never train on it.**

In [ ]:
s = SESSIONS[0]
MEANING = {
    "ts": "unix time of THIS sample (s)",
    "label": "what the operator was doing",
    "session_type": "fast (2-3s cycles) or slow (15-20s ramps)",
    "experiment_start_ts": "start of the labelled segment this sample belongs to",
    **{a: ("force, N" if a[0]=="f" else "torque, N*m") + " -- uFactory frame" for a in AXES},
    **{c: "raw ADC counts, one Hall channel (unitless)" for c in CHAN_COLS},
}
for name in ("ufactory_raw","ufactory_calibrated","homemade_raw","homemade_calibrated"):
    cols = list(stream(s,name)[0].keys())
    print(f"\n{name}.csv  ({len(cols)} cols)")
    for c in cols: print(f"    {c:<22} {MEANING.get(c,'?')}")

## Step 3 — Synchronisation

Both streams are timestamped by the same collector process, so they *should* already align —
but "should" isn't evidence. We sweep an artificial lag and find where a linear fit between
channels and wrench is best. If the optimum is at ~0, no correction is needed.

In [ ]:
def best_lag(session, lags=np.arange(-0.4, 0.41, 0.05)):
    uf, hm = stream(session,"ufactory_calibrated"), stream(session,"homemade_raw")
    ut, uy = ts_of(uf), arr(uf, AXES)
    ht, X  = ts_of(hm), arr(hm, CHAN_COLS)
    Xc = X - X.mean(axis=0)
    A  = np.hstack([Xc, np.ones((len(Xc),1))])
    out = []
    for lag in lags:
        y = np.column_stack([np.interp(ht+lag, ut, uy[:,i]) for i in range(6)])
        y = y - y.mean(axis=0)
        W,*_ = np.linalg.lstsq(A, y, rcond=None)
        out.append((1 - (y - A@W).var(axis=0)/np.maximum(y.var(axis=0),1e-12)).mean())
    return lags, np.array(out)

fig, ax = plt.subplots(figsize=(7,3.5))
for s in SESSIONS:
    lags, r2 = best_lag(s)
    ax.plot(lags, r2, alpha=.6, lw=1)
    print(f"{s}: best lag = {lags[np.argmax(r2)]:+.2f}s")
ax.axvline(0, color="k", ls="--", lw=1)
ax.set_xlabel("assumed lag (s)"); ax.set_ylabel("mean R² of a linear fit")
ax.set_title("Alignment check — peak at 0 means the clocks agree")
plt.tight_layout(); plt.show()

## Step 4 — Sampling rates

The two sensors run at different rates (the MCU sets the homemade rate; our polling loop sets
the uFactory one). That's fine, but it dictates one rule:

> **Always interpolate the faster stream onto the slower one's timestamps.**
> Never upsample the target you are trying to predict.

In [ ]:
print(f"{'session':>12} {'sensor':>10} {'mean Hz':>9} {'median dt':>11} {'p99 dt':>9} {'gaps>3x':>8}")
for s in SESSIONS:
    for label, name in (("uFactory","ufactory_calibrated"), ("homemade","homemade_raw")):
        t  = ts_of(stream(s,name)); dt = np.diff(t); med = np.median(dt)
        print(f"{s:>12} {label:>10} {len(t)/(t[-1]-t[0]):9.1f} {med*1000:9.1f}ms "
              f"{np.percentile(dt,99)*1000:8.1f}ms {(dt>3*med).sum():8d}")

## Step 5 — Data quality

NaNs, duplicate timestamps, out-of-order samples, and **stuck channels**
(zero variance across a whole session = a dead sensor).

In [ ]:
print(f"{'session':>12} {'stream':>21} {'rows':>8} {'NaN':>5} {'dup_ts':>7} {'non-mono':>9} {'stuck':>6}")
for s in SESSIONS:
    for name in ("ufactory_raw","ufactory_calibrated","homemade_raw","homemade_calibrated"):
        rows = stream(s,name)
        cols = CHAN_COLS if name=="homemade_raw" else AXES
        v, t = arr(rows, cols), ts_of(rows)
        stuck = sum(1 for j in range(v.shape[1]) if v[:,j].std()==0)
        print(f"{s:>12} {name:>21} {len(rows):8d} {int(np.isnan(v).sum()):5d} "
              f"{len(t)-len(np.unique(t)):7d} {int((np.diff(t)<0).sum()):9d} {stuck:6d}")

## Step 6 — Unloaded periods

Every offset correction depends on these, so they have to be *genuinely* unloaded.
`|F| spread during rest` measures variation **around** the resting mean — if it's large,
the operator was still touching the sensor and that segment isn't a true zero.

In [ ]:
print(f"{'session':>12} {'segments':>9} {'samples':>8} {'% of run':>9} {'|F| spread @rest (p95)':>24}")
for s in SESSIONS:
    uf  = stream(s,"ufactory_calibrated")
    y   = arr(uf, AXES)
    lab = np.array([r["label"] for r in uf])
    seg = np.array([float(r["experiment_start_ts"]) for r in uf])
    rest = lab=="resting"
    f = np.linalg.norm(y[rest][:,:3] - y[rest][:,:3].mean(axis=0), axis=1)
    print(f"{s:>12} {len(np.unique(seg[rest])):9d} {rest.sum():8d} {100*rest.mean():8.0f}% "
          f"{np.percentile(f,95):19.2f} N")

## Step 7 — Offsets and drift

Zero point per resting segment, so drift *within* a run is visible.

If drift is larger than the sensor's own noise, a single constant zero is not enough —
which is exactly why the fitting pipeline interpolates the baseline between resting segments.

In [ ]:
def rest_baselines(session):
    """Zero points and their times, from each resting segment."""
    hm  = stream(session,"homemade_raw")
    X, t = arr(hm, CHAN_COLS), ts_of(hm)
    lab = np.array([r["label"] for r in hm])
    seg = np.array([float(r["experiment_start_ts"]) for r in hm])
    rest = lab=="resting"
    groups = {}
    for i in np.where(rest)[0]: groups.setdefault(seg[i], []).append(i)
    keys = sorted(groups)
    centres = np.array([t[groups[k]].mean() for k in keys])
    base    = np.array([X[groups[k]].mean(axis=0) for k in keys])
    return centres, base, X[rest].std(axis=0)

fig, ax = plt.subplots(figsize=(8,4))
for s in SESSIONS:
    c, b, noise = rest_baselines(s)
    drift = np.abs(b - b[0]).max()
    ax.plot(c - c[0], (b - b[0])[:,0], marker="o", ms=3, lw=1, alpha=.7)
    print(f"{s}: {len(c):2d} zero points | noise {noise.mean():5.2f} counts | "
          f"max drift {drift:5.2f} counts ({drift/max(noise.mean(),1e-9):4.1f}x noise)")
ax.set_xlabel("time into session (s)"); ax.set_ylabel("ch1 zero drift (counts)")
ax.set_title("Zero point wanders during a run"); ax.axhline(0, color="k", lw=.8)
plt.tight_layout(); plt.show()

## Step 8 — After offset removal

Build the corrected arrays we'll actually fit on, then confirm that zero load now reads zero.

This is the function the training pipeline uses; everything after this point consumes its output.

In [ ]:
def load_corrected(session, frame_offset_m=0.0):
    """Return (X, y) with a time-varying baseline removed from both.

    frame_offset_m: shift the uFactory wrench to a point `d` metres along +Z, i.e.
    express it about the homemade sensor's own origin. M' = M - r x F with r=(0,0,d):
        mx' = mx + d*fy ,  my' = my - d*fx ,  mz' = mz
    Leave 0.0 to keep the uFactory's own frame.
    """
    uf, hm = stream(session,"ufactory_calibrated"), stream(session,"homemade_raw")
    ut, uy = ts_of(uf), arr(uf, AXES)
    ht, X  = ts_of(hm), arr(hm, CHAN_COLS)
    lab = np.array([r["label"] for r in hm])
    seg = np.array([float(r["experiment_start_ts"]) for r in hm])

    # interpolate the FASTER uFactory stream onto the SLOWER homemade timestamps
    y = np.column_stack([np.interp(ht, ut, uy[:,i]) for i in range(6)])
    if frame_offset_m:
        d = frame_offset_m
        y = np.column_stack([y[:,0], y[:,1], y[:,2],
                             y[:,3] + d*y[:,1], y[:,4] - d*y[:,0], y[:,5]])

    rest = lab=="resting"
    groups = {}
    for i in np.where(rest)[0]: groups.setdefault(seg[i], []).append(i)
    keys = sorted(groups)
    centres = np.array([ht[groups[k]].mean() for k in keys])
    bx = np.array([X[groups[k]].mean(axis=0) for k in keys])
    by = np.array([y[groups[k]].mean(axis=0) for k in keys])
    Xb = np.column_stack([np.interp(ht, centres, bx[:,j]) for j in range(CHANNELS)])
    yb = np.column_stack([np.interp(ht, centres, by[:,j]) for j in range(6)])
    return X - Xb, y - yb, lab

print(f"{'session':>12} {'homemade resid @rest':>22} {'uFactory resid @rest':>22}")
for s in SESSIONS:
    Xc, yc, lab = load_corrected(s)
    r = lab=="resting"
    print(f"{s:>12} {np.abs(Xc[r]).mean():16.3f} counts "
          f"{np.linalg.norm(yc[r][:,:3],axis=1).mean():16.3f} N")

## Step 9 — Ranges actually used

Two separate questions: how much of the **reference's** range did we exercise (does it
cover what we need?), and how much of the **homemade sensor's** ADC band did the signal
occupy (this sets its resolution floor).

In [ ]:
X = np.vstack([stream_arr for stream_arr in (arr(stream(s,"homemade_raw"), CHAN_COLS) for s in SESSIONS)])
Y = np.vstack([arr(stream(s,"ufactory_calibrated"), AXES) for s in SESSIONS])

print("uFactory (reference) -- excitation vs its own rated range:")
for i,a in enumerate(AXES):
    mx = np.abs(Y[:,i]).max(); pct = 100*mx/UF_RATED[a]
    print(f"  {a}: max {mx:7.2f} of {UF_RATED[a]:6.1f} rated = {pct:5.1f}%"
          + ("   <-- OVER RATED RANGE" if pct>100 else ""))

p2p = X.max(axis=0) - X.min(axis=0)
band = OPENFT_VALID[1]-OPENFT_VALID[0]
print(f"\nhomemade raw channels (firmware valid band {OPENFT_VALID[0]:.0f}..{OPENFT_VALID[1]:.0f}):")
print(f"  observed min {X.min():.0f}, max {X.max():.0f}")
print(f"  per-channel peak-to-peak: mean {p2p.mean():.1f}, max {p2p.max():.1f} counts")
print(f"  = {100*p2p.mean()/band:.2f}% of the usable ADC band")

fig, ax = plt.subplots(figsize=(8,3))
ax.bar(range(1,17), p2p, color="#1f6f8b")
ax.set_xlabel("channel"); ax.set_ylabel("peak-to-peak (counts)")
ax.set_title("How far each channel actually moved across all sessions")
plt.tight_layout(); plt.show()

## Step 10 — Do the two sensors share a coordinate frame?

This is the one you're working on physically, and the data can answer part of it.

**Axis alignment** and **origin offset** are separate questions:

- *Alignment* — do +X, +Y, +Z point the same way on both sensors? If yes, a force along the
  uFactory's +X should show up as a force along the homemade sensor's +X.
- *Origin offset* — the two sensors sit at different points along the tool axis, so they report
  **different torques for the same load**: `M_uf = M_diy + r × F`, where `r` is the vector between
  their origins. With `r = (0,0,d)` that means `my_uf = my_diy + d·fx` and `mx_uf = mx_diy − d·fy`.

The offset is measurable from the data: sweep `d` and find where the artificial correlation
between force and torque disappears. Compare that to your CAD numbers as an independent check.

In [ ]:
Y = np.vstack([arr(stream(s,"ufactory_calibrated"), AXES) for s in SESSIONS])

def shift_frame(y, d):
    return np.column_stack([y[:,0], y[:,1], y[:,2],
                            y[:,3] + d*y[:,1], y[:,4] - d*y[:,0], y[:,5]])

ds = np.arange(0, 0.121, 0.0025)
score = []
for d in ds:
    ys = shift_frame(Y, d)
    score.append((abs(np.corrcoef(ys[:,0],ys[:,4])[0,1]) + abs(np.corrcoef(ys[:,1],ys[:,3])[0,1]))/2)
score = np.array(score)
d_fit = ds[np.argmin(score)]

# --- your CAD numbers ---
UF_FLANGE_TO_SENSOR_MM  = 49.2    # uFactory mounting flange -> uFactory sensor origin
DIY_BACKPLATE_TO_SENS_MM = 7.25   # DIY CNC backplate -> DIY sensor origin
d_cad = (UF_FLANGE_TO_SENSOR_MM + DIY_BACKPLATE_TO_SENS_MM)/1000

fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(ds*1000, score, color="#1f6f8b")
ax.axvline(d_fit*1000, color="#2a7350", ls="--", label=f"data-fitted {d_fit*1000:.1f} mm")
ax.axvline(d_cad*1000, color="#a2383b", ls=":",  label=f"CAD {d_cad*1000:.2f} mm")
ax.set_xlabel("assumed origin offset d (mm)")
ax.set_ylabel("mean |corr| between force and torque")
ax.set_title("Origin offset between the two sensors"); ax.legend()
plt.tight_layout(); plt.show()

print(f"data-fitted offset : {d_fit*1000:.1f} mm")
print(f"CAD  (49.2 + 7.25) : {d_cad*1000:.2f} mm")
print(f"disagreement       : {abs(d_fit-d_cad)*1000:.2f} mm")

### Axis alignment check

If the two sensors' axes are aligned, pushing along one uFactory axis should move a
*consistent* set of homemade channels, and the sign should be stable across sessions.
This doesn't prove alignment on its own — it's a cross-check against the physical
measurement you're making.

In [ ]:
# For each uFactory axis, which homemade channels respond most, and with what sign?
Xs, Ys = [], []
for s in SESSIONS:
    Xc, yc, lab = load_corrected(s)
    Xs.append(Xc); Ys.append(yc)
Xc, yc = np.vstack(Xs), np.vstack(Ys)

print("Correlation of each homemade channel with each uFactory axis")
print("    " + "".join(f"{a:>7}" for a in AXES))
for j in range(CHANNELS):
    r = [np.corrcoef(Xc[:,j], yc[:,i])[0,1] for i in range(6)]
    print(f"ch{j+1:<2}" + "".join(f"{v:7.2f}" for v in r))

fig, ax = plt.subplots(figsize=(7,5))
M = np.array([[np.corrcoef(Xc[:,j], yc[:,i])[0,1] for i in range(6)] for j in range(CHANNELS)])
im = ax.imshow(M, cmap="RdBu_r", vmin=-.8, vmax=.8, aspect="auto")
ax.set_xticks(range(6), AXES); ax.set_yticks(range(16), [f"ch{i+1}" for i in range(16)])
ax.set_title("Which channels see which axis"); plt.colorbar(im, label="correlation")
plt.tight_layout(); plt.show()

---

## Where this leaves us

Once every step above is understood and passing, `load_corrected()` is the function that
feeds the fit. Two open decisions it exposes:

1. **Which frame should the calibration target?**
   - `frame_offset_m=0` → the homemade sensor learns to report *what the uFactory reports*.
     Good for a drop-in replacement, but the model can absorb the fixed `d·F` term rather than
     learning real torque sensitivity.
   - `frame_offset_m=d` → the homemade sensor learns the wrench about **its own origin**,
     which is the physically honest target. Harder, and it reveals how little genuine torque
     sensitivity the sensor has.

2. **Whether the loading covers enough variety** — if every push has the same force-to-torque
   ratio, no model can separate the two, whichever frame you choose.

Fit with `fit_calibration.py` once you're satisfied with the answers here.